## 5. Features Comportamentais (Histórico)
Objetivo: Criar features comportamentais baseadas no histórico do cliente, com foco em capturar padrões de inadimplência ao longo do tempo.

Ideia: A premissa central é que o comportamento passado do cliente é um forte preditor do comportamento futuro, especialmente em problemas de crédito.

Decisão: Todas as variáveis foram construídas utilizando apenas informações passadas, através do uso de shift(1), garantindo que não haja vazamento de informação (data leakage) e simulando corretamente um cenário de produção.

Tipos de features criadas:

• Lags de inadimplência: Capturam o comportamento recente do cliente (ex: se inadimpliu nos últimos períodos).

• Métricas de curto prazo (rolling): Médias móveis de inadimplência e atraso (3 períodos), capturando tendência recente.

• Métricas de longo prazo: Janelas maiores (6 períodos) e médias acumuladas (expanding), capturando padrão estrutural de comportamento.

• Perfil histórico: Taxas médias, atraso médio e volume de cobranças refletem o perfil de risco do cliente ao longo do tempo.

• Valor financeiro: O valor médio pago pelo cliente foi incluído para capturar possível relação entre ticket médio e inadimplência.

• Cliente novo: A flag identifica clientes sem histórico, diferenciando ausência de informação de comportamento efetivamente adimplente.
Importância: Essa abordagem permite combinar sinais de curto e longo prazo, aumentando o poder preditivo do modelo e mantendo consistência temporal.

## 1 SETUP feature_engineering

In [ ]:
# ============================================================
# SETUP — deve ser a primeira célula do notebook
# ============================================================
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # raiz do projeto no PATH

import pandas as pd
import numpy as np
from src.features import build_behavioral_features, build_context_features, impute_missing, COLS_LAG

PATH = '../data/'

# Carregamento
cadastral = pd.read_csv(PATH + 'base_cadastral.csv', sep=';')
info      = pd.read_csv(PATH + 'base_info.csv', sep=';')
pag       = pd.read_csv(PATH + 'base_pagamentos_desenvolvimento.csv', sep=';')

# Limpeza de colunas
for base in [cadastral, info, pag]:
    base.columns = base.columns.str.strip()

# Datas
pag['DATA_PAGAMENTO']        = pd.to_datetime(pag['DATA_PAGAMENTO'], errors='coerce')
pag['DATA_VENCIMENTO']       = pd.to_datetime(pag['DATA_VENCIMENTO'], errors='coerce')
pag['DATA_EMISSAO_DOCUMENTO']= pd.to_datetime(pag['DATA_EMISSAO_DOCUMENTO'], errors='coerce')

# Target
pag['dias_atraso'] = (pag['DATA_PAGAMENTO'] - pag['DATA_VENCIMENTO']).dt.days
pag['dias_atraso'] = pag['dias_atraso'].fillna(999)
pag['target']      = (pag['dias_atraso'] >= 5).astype(int)

# Base final
df = pag.merge(info, on=['ID_CLIENTE', 'SAFRA_REF'], how='left')
df = df.merge(cadastral, on='ID_CLIENTE', how='left')
df = df.sort_values(['ID_CLIENTE', 'SAFRA_REF', 'DATA_VENCIMENTO']).reset_index(drop=True)

# CUTOFF temporal (últimos 20% das safras)
CUTOFF = df['DATA_VENCIMENTO'].quantile(0.8)

print(f"Base carregada: {df.shape}")
print(f"CUTOFF: {CUTOFF.date()}")

Base carregada: (77414, 18)
CUTOFF: 2021-01-06


In [ ]:
# Garantir ordenação correta antes de qualquer lag/rolling
df = df.sort_values(['ID_CLIENTE', 'SAFRA_REF', 'DATA_VENCIMENTO']).reset_index(drop=True)

## Lags de inadimplência 

# shift(1) dentro do groupby → só enxerga o passado do cliente
df['target_lag_1'] = df.groupby('ID_CLIENTE')['target'].shift(1)
df['target_lag_2'] = df.groupby('ID_CLIENTE')['target'].shift(2)
df['target_lag_3'] = df.groupby('ID_CLIENTE')['target'].shift(3)

# Rolling dentro do groupby (FIX: bug silencioso)
 
df['inad_ult_3'] = (
    df.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
 
df['atraso_ult_3'] = (
    df.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
 
# Média de atraso em janela de 6 meses (captura tendência de longo prazo)
df['atraso_ult_6'] = (
    df.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)

# Taxa histórica global por cliente 
# expanding() = média acumulada de todas as safras anteriores
df['taxa_hist_inad'] = (
    df.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).expanding().mean())
)
 
df['atraso_hist_medio'] = (
    df.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).expanding().mean())
)
 
# Total de cobranças anteriores do cliente
df['n_cobrancas_hist'] = (
    df.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).expanding().count())
)

 
df['valor_medio_cliente'] = (
    df.groupby('ID_CLIENTE')['VALOR_A_PAGAR']
    .transform(lambda x: x.shift(1).expanding().mean()) #usa só o passado
)
 
 
# Flag de cliente novo
# Criada ANTES do fillna para preservar a informação de ausência
# (cliente novo ≠ cliente com histórico limpo)
df['flag_cliente_novo'] = df['target_lag_1'].isna().astype(int)

## 5.1 Features de Contexto (Tempo e Cobrança)
Objetivo: Criar features de contexto relacionadas ao tempo de relacionamento e às características da cobrança atual.

Ideia: Além do histórico comportamental, o contexto em que a cobrança ocorre pode influenciar diretamente a probabilidade de inadimplência.

Decisão:

• Tempo de relacionamento: A variável 'tempo_cliente' captura o tempo desde o cadastro até a data da cobrança. Clientes mais antigos tendem a ter comportamento mais estável, enquanto clientes recentes podem apresentar maior incerteza de risco.

• Prazo da cobrança: A variável 'dias_prazo_cobranca' representa o tempo entre emissão e vencimento podendo influenciar a capacidade do cliente de se organizar para pagamento.

• Sazonalidade: As variáveis 'mes_safra' e 'trimestre_safra' capturam padrões sazonais, permitindo ao modelo identificar períodos com maior propensão à inadimplência
Importância: Essas features adicionam uma visão contextual ao modelo, complementando o histórico do cliente com informações do momento da cobrança.


In [ ]:
# Tempo de relacionamento
df['DATA_CADASTRO'] = pd.to_datetime(df['DATA_CADASTRO'], errors='coerce')
df['tempo_cliente']  = (df['DATA_VENCIMENTO'] - df['DATA_CADASTRO']).dt.days
 
 
#  Features da cobrança atual
df['DATA_EMISSAO_DOCUMENTO'] = pd.to_datetime(df['DATA_EMISSAO_DOCUMENTO'], errors='coerce')
df['dias_prazo_cobranca']    = (df['DATA_VENCIMENTO'] - df['DATA_EMISSAO_DOCUMENTO']).dt.days
df['mes_safra']              = pd.to_datetime(df['SAFRA_REF']).dt.month
df['trimestre_safra']        = pd.to_datetime(df['SAFRA_REF']).dt.quarter

## 5.2 Tratamento de Missing Values
Objetivo: Tratar valores ausentes de forma consistente com o significado de cada variável, evitando distorções no modelo.

Estratégia:

• Variáveis históricas → imputadas com 0 (interpretado como ausência de histórico do cliente)

• Variáveis numéricas → imputadas com mediana (método robusto a outliers e distribuição assimétrica)

• Variáveis categóricas → imputadas com 'DESCONHECIDO' (preserva informação de ausência sem introduzir viés artificial)
Importância: O tratamento adequado de valores ausentes evita perda de informação e garante maior estabilidade e performance do modelo.

In [ ]:
COLS_LAG = [
    'target_lag_1', 'target_lag_2', 'target_lag_3',
    'inad_ult_3', 'atraso_ult_3', 'atraso_ult_6',
    'taxa_hist_inad', 'atraso_hist_medio', 'n_cobrancas_hist',
    'valor_medio_cliente',
]
df[COLS_LAG] = df[COLS_LAG].fillna(0)

# Medianas calculadas apenas sobre o treino (CUTOFF definido na célula 02)
df_train_ref = df[df['DATA_VENCIMENTO'] < CUTOFF]

num_cols = [
    c for c in df.select_dtypes(include='number').columns
    if c not in COLS_LAG and df[c].isna().any()
]
train_medians = df_train_ref[num_cols].median()

for col in num_cols:
    df[col] = df[col].fillna(train_medians[col])

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].fillna('DESCONHECIDO')

print(f"Medianas calculadas sobre {len(df_train_ref):,} registros de treino")
print(f"Colunas imputadas: {num_cols}")

Medianas calculadas sobre 61,888 registros de treino
Colunas imputadas: ['VALOR_A_PAGAR', 'RENDA_MES_ANTERIOR', 'NO_FUNCIONARIOS']


## 5.3 Validação Final
Objetivo: Validar a correta construção das features e garantir consistência antes da modelagem.

Validações realizadas:

• Estrutura da base: Confirmação do tamanho final após criação das variáveis.

• Inspeção manual: Visualização de amostras das principais features para verificar coerência dos valores,
               especialmente variáveis baseadas em histórico (lags, rolling e acumuladas).

• Consistência temporal: Verificação indireta de que as variáveis utilizam apenas informações passadas, garantindo ausência de data leakage.
Importância: Essa etapa reduz o risco de erros silenciosos na engenharia de features, que poderiam comprometer a performance e a confiabilidade do modelo.

In [ ]:
print("Features criadas sem data leakage - OK")
print(f"   Shape: {df.shape}")

print(df[['target_lag_1', 'inad_ult_3', 'atraso_ult_3',
          'taxa_hist_inad', 'valor_medio_cliente',
          'flag_cliente_novo']].head(8).to_string())
 

Features criadas sem data leakage - OK
   Shape: (77414, 33)
   target_lag_1  inad_ult_3  atraso_ult_3  taxa_hist_inad  valor_medio_cliente  flag_cliente_novo
0           0.0         0.0      0.000000             0.0             0.000000                  1
1           0.0         0.0      0.000000             0.0        100616.100000                  0
2           0.0         0.0      0.000000             0.0         97339.450000                  0
3           0.0         0.0     -0.666667             0.0         99121.666667                  0
4           0.0         0.0     -0.666667             0.0         96729.450000                  0
5           0.0         0.0     -0.666667             0.0         87662.160000                  0
6           0.0         0.0      0.000000             0.0         82986.926667                  0
7           0.0         0.0      0.000000             0.0         76759.945714                  0
